# Encapsulation and Shared References

In this lesson, you will protect object state, trace shared references, and distinguish an object's values from values shared by its class.

CSC-239 · Module 5 · Lesson 2 of 3

You have defined classes, initialized objects, and directed method calls to separate objects. Now you will control which changes callers may request and follow what happens when two variables reach the same object. You will finish by building two protected seat pools and testing requests at their limits.

The examples use the variables, constructors, instance methods, Boolean conditions, and early returns taught earlier. The [Module 5 glossary](terms.md) supports the explanations below.


## Learning Goals

- Protect a state rule with private fields and guarded public operations.
- Distinguish reference reassignment, shared-object mutation and class-wide state.


## Why This Matters

An application may let several features work with the same account, reservation, or equipment record. Those features need consistent rules for changing it. If every caller assigns field values directly, one overlooked check can leave the record in a state that other parts of the application cannot use correctly.

Protected operations put the checks next to the state they govern. Understanding shared references then helps you explain why a change made in one feature appears in another. Separating object state from class state also prevents one record's value from accidentally becoming everyone's value. These ideas prepare you to implement a collection with private storage and later connect a user interface to a reliable model.


## Check Your Starting Point

Explain why two separate `new` expressions create two objects, while two variable names alone do not prove that there are two objects. Recall what happens when a method assigns a new number to an `int` parameter. Finally, explain how an early `return false` prevents later statements in that call from running. Record your explanation before opening the answer.


In [ ]:
Your response:

Two new expressions versus two variable names:

Changing an int parameter versus its caller variable:

What an early return prevents:


<details>
<summary>Show answer</summary>

Each executed new expression creates an object. A variable stores a value; merely declaring another variable does not create another object. The last lesson used two new expressions to give two passes independent state.

An int parameter receives a copy of the argument's numeric value. Assigning another number to that local parameter does not assign the caller's variable. An early return ends the current method call, so statements after it do not run on that path. We can use that control-flow behavior to reject a request before a state-changing assignment.

</details>


## Video Demonstration

Watch a valid request change one bin and an excessive request leave it unchanged. Distinguish the stock in each object from the single count stored by the class.

<video controls preload="metadata" width="960">
  <source src="media/02_encapsulation_and_shared_references/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/02_encapsulation_and_shared_references/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the encapsulation and shared references demonstration transcript](media/02_encapsulation_and_shared_references/transcript.md).


## Concept

### Give out items without losing track of the stock

A campus gallery lends small supply bins to activity tables. A bin stores a whole-number count of available items. A caller may request a positive number of items, but may not take more than the bin currently holds. An accepted request reduces that bin's stock. A rejected request must leave it unchanged. The caller needs both a success result and a way to read the remaining count.

Our initial counts are nonnegative and fit in `int`. We first teach the guard with a bin containing four items. Later examples separately examine two references to one object, a helper's reference parameter, and a construction count shared by a class.


### Check a request before changing stored state

The earlier pass class let surrounding lesson code read its fields directly. The bin needs a boundary around changes. **Encapsulation** means grouping an object's state with operations that control how callers use and change it. Callers should request items through an operation that checks the request.

**Access control** determines which code may use a member. In the top-level class shown here, the Java keyword **`private`** keeps the field accessible within its defining class:

```java
private int stock;
```

The count is still an instance field: every bin has its own value. The access modifier does not make it shared. Other lesson code cannot directly assign `bin.stock = -1`. That expression is an intentionally invalid access example, not a statement to add to a runnable cell.

The constructor establishes the starting count:

```java
public GalleryBin(int stock) {
    this.stock = stock;
}
```

The Java keyword **`public`** makes this constructor an intended entry point for callers that can access the class. `this.stock` selects the new object's field; the unqualified `stock` selects the constructor parameter. The caller supplies an initially nonnegative count. That requirement is a **precondition**, a rule the caller must satisfy before the operation starts. This constructor does not check the rule. Use only valid starting counts here; a later exception lesson will teach how a constructor can reject invalid input.

A **state invariant** is a rule that should hold for the object after construction and after its operations. Our rule is `stock >= 0`. The public `take` method must preserve it while also rejecting requests for zero or negative amounts:

```java
public boolean take(int amount) {
    if (amount <= 0 || amount > stock) {
        return false;
    }
    stock = stock - amount;
    return true;
}
```

The method receives the requested number of items in `amount`. Its `boolean` result tells the caller whether the request succeeded. The condition combines two reasons to reject with `||`, the logical OR operator: the amount is not positive, or it exceeds the available stock. If either reason applies, `return false` ends this call immediately. The subtraction below it never runs.

Only an accepted request reaches `stock = stock - amount`. That assignment reduces the receiving bin's count. `return true` then reports success. An exact request for all remaining items is valid: it leaves zero, which satisfies the invariant. Making the field private is useful, but the method's correct order of checks and updates is what preserves this rule. A private field can still be damaged by an incorrect method inside its class.

Callers also need to inspect the result without changing it:

```java
public int getStock() {
    return stock;
}
```

This reader returns the current integer count. It exposes a useful operation, not direct permission to assign the private field. In the complete example, each report calls `take` and then `getStock`. Java evaluates the first call before the later call in the concatenation, so the reported stock is the count after that request. Checking both the Boolean result and the stored count helps reveal a bug that returns `false` after already changing the state.


In [ ]:
class GalleryBin {
    private int stock;
    public GalleryBin(int stock) { this.stock = stock; }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) { return false; }
        stock = stock - amount;
        return true;
    }
    public int getStock() { return stock; }
}
GalleryBin bin = new GalleryBin(4);
System.out.println("Take 2: " + bin.take(2) + ", stock " + bin.getStock());
System.out.println("Take 3: " + bin.take(3) + ", stock " + bin.getStock());
System.out.println("Take 0: " + bin.take(0) + ", stock " + bin.getStock());
System.out.println("Take -2: " + bin.take(-2) + ", stock " + bin.getStock());
System.out.println("Take 2: " + bin.take(2) + ", stock " + bin.getStock());


Expected output:

```text
Take 2: true, stock 2
Take 3: false, stock 2
Take 0: false, stock 2
Take -2: false, stock 2
Take 2: true, stock 0
```

The first request takes two of the four items and leaves two. Requests for three, zero, and negative two each fail without changing that count. The final request for exactly two succeeds and leaves zero. A Boolean result alone would not prove unchanged state; each report reads both.


<details class="animation-panel" open>
<summary>Show or hide animation: guard before state change</summary>
<img src="media/02_encapsulation_and_shared_references/guard_before_state_change.gif" alt="Trace accepted and rejected requests against a nonnegative stock invariant. Exact remaining request 2 passes; stock becomes 0. Print Take 2: true, stock 0. Every rejection preserved state." width="960" style="max-width:100%;height:auto;">
</details>

The animation follows the state changes in the complete example above. Use **Show or hide animation** to stop viewing its motion, or [view the guard before state change still image](media/02_encapsulation_and_shared_references/guard_before_state_change_still.png) for the final state. The reading and output explanation provide the same reasoning without animation.


### Distinguish another name from another object

A protected object can still be shared. Imagine two parts of an application working with one exhibit's visitor counter. They should observe the same count. An **alias** is another reference that reaches the same object; **aliasing** is this shared-reference relationship.

The complete example below defines `ExhibitCounter` with a private count, a constructor, an `addOne` operation, and a `getCount` reader. These use the same access-control pattern just explained. The important difference lies in the caller:

```java
ExhibitCounter original = new ExhibitCounter(5);
ExhibitCounter alias = original;
alias.addOne();
```

The first line creates one counter with an initial count of five. The second line copies the reference value stored in `original` into another variable. It contains no `new` expression, so it creates no second counter and copies no fields. Both variables now reach the same object.

The call through `alias` selects that shared object as the receiver. `addOne` updates its count from five to six. Reading through `original` therefore sees six too; there is no separate count attached to the variable name `original`.

Now compare a separate creation:

```java
ExhibitCounter separate = new ExhibitCounter(5);
```

This line has its own `new` expression. It creates another object whose count starts at five. It does not copy the updated state of the first counter. In the complete example, the two shared references report six and the separate reference reports five. Count object creations and follow reference values when tracing state; counting variable names alone gives the wrong model.


In [ ]:
class ExhibitCounter {
    private int count;
    public ExhibitCounter(int count) { this.count = count; }
    public void addOne() { count = count + 1; }
    public int getCount() { return count; }
}
ExhibitCounter original = new ExhibitCounter(5);
ExhibitCounter alias = original;
alias.addOne();
ExhibitCounter separate = new ExhibitCounter(5);
System.out.println("Original: " + original.getCount());
System.out.println("Alias: " + alias.getCount());
System.out.println("Separate: " + separate.getCount());


Expected output:

```text
Original: 6
Alias: 6
Separate: 5
```

The reference-copy assignment gives original and alias access to one counter. Updating through alias changes that counter to six, which both readers observe. A separate new expression creates another counter at five. Its field is independent even though the class definition is the same.


<details class="animation-panel" open>
<summary>Show or hide animation: aliases reach one object</summary>
<img src="media/02_encapsulation_and_shared_references/aliases_reach_one_object.gif" alt="Distinguish copying a reference from creating an additional object. Print Original: 6, Alias: 6, Separate: 5. Two references share A; a new expression created independent B." width="960" style="max-width:100%;height:auto;">
</details>

The animation follows the state changes in the complete example above. Use **Show or hide animation** to stop viewing its motion, or [view the aliases reach one object still image](media/02_encapsulation_and_shared_references/aliases_reach_one_object_still.png) for the final state. The reading and output explanation provide the same reasoning without animation.


### Follow the reference copied into a helper

Sharing also occurs when a caller passes an object reference to a method. **Mutation** means changing an existing object's state. Java passes arguments by value, including reference values: a parameter receives a copy of the supplied reference. That copy can reach the same object as the caller's variable.

The first helper below accepts a reference to an `ExhibitCounter`:

```java
static void change(ExhibitCounter item) {
    item.addOne();
}
```

`item` is a parameter local to this call. Its declared type tells us it can refer to an ExhibitCounter. Calling `ExhibitTools.change(original)` copies the reference from `original` into `item`. Both then reach the counter that started at five. `item.addOne()` mutates that counter to six. When the helper ends, the object still contains six, and the caller can read it through `original`. The `void` method returns no value; its useful effect is the lasting change to the object.

A different helper assigns a new reference to its parameter:

```java
static void replace(ExhibitCounter item) {
    item = new ExhibitCounter(12);
}
```

At the start of this call, `item` again receives a copy of the caller's reference. The `new` expression then creates a different counter containing twelve. The assignment stores that new reference only in the local parameter `item`. It does not assign the caller's variable `original`, and it does not assign a field in the original object.

The complete example calls `change` first and `replace` second. The report after `change` is six because the shared object's field changed. The report after `replace` is still six because the caller continues to reach that same object. The new counter's twelve is never copied back into it.

This extends the earlier distinction between changing an `int` parameter and changing an object's field. Reassigning a parameter changes its local value. Calling a mutating method through a copied reference can change an object outside that local variable. Do not describe this as Java passing objects by reference: both helpers receive a copied value.


In [ ]:
class ExhibitCounter {
    private int count;
    public ExhibitCounter(int count) { this.count = count; }
    public void addOne() { count = count + 1; }
    public int getCount() { return count; }
}
class ExhibitTools {
    static void change(ExhibitCounter item) { item.addOne(); }
    static void replace(ExhibitCounter item) { item = new ExhibitCounter(12); }
}
ExhibitCounter original = new ExhibitCounter(5);
ExhibitTools.change(original);
System.out.println("After change: " + original.getCount());
ExhibitTools.replace(original);
System.out.println("After replace: " + original.getCount());


Expected output:

```text
After change: 6
After replace: 6
```

The change helper reaches the caller's counter through a copied reference and increments its field to six. The replace helper assigns a new reference only to its local parameter. The caller still reaches the counter containing six, so both reports read six.


<details class="animation-panel" open>
<summary>Show or hide animation: parameter reference and object state</summary>
<img src="media/02_encapsulation_and_shared_references/parameter_reference_and_object_state.gif" alt="A helper changes the shared counter from five to six. Another helper creates a counter at twelve and assigns it only to its local parameter. The caller still reads six from the original counter." width="960" style="max-width:100%;height:auto;">
</details>

The animation follows the state changes in the complete example above. Use **Show or hide animation** to stop viewing its motion, or [view the parameter reference and object state still image](media/02_encapsulation_and_shared_references/parameter_reference_and_object_state_still.png) for the final state. The reading and output explanation provide the same reasoning without animation.


### Keep a construction count separate from each bin's stock

The gallery may need a report of how many bin objects the application has constructed. That total belongs to the class-wide record. It should not be stored as a separate running total inside every bin.

A **static field** belongs to the class, rather than separately to each object. The Java keyword **`static`** marks that distinction:

```java
private int stock;
private static int created = 0;
```

Every `CountedGalleryBin` object has its own `stock` field. The class has one shared `created` field. Its initializer starts the class-wide count at zero when this class is initialized. The constructor performs two different jobs:

```java
public CountedGalleryBin(int stock) {
    this.stock = stock;
    created = created + 1;
}
```

The first assignment initializes the new object's stock from this call's argument. The second increments the shared construction count. Constructing an eight-item bin and then a three-item bin leaves two separate stock values and a shared count of two. Copying a reference into an alias does not call the constructor, so it does not increment that count.

A static reader reports the class value:

```java
public static int getCreated() {
    return created;
}
```

Call it through the class name as `CountedGalleryBin.getCreated()`. This method does not need a particular bin as its receiver. A static method has no current instance and cannot use `this` to choose one object's stock. The separate instance reader `getStock()` still needs a receiver such as `first` or `second`.

In the complete example, `alias` receives the reference from `first`. Taking two through that alias changes the first bin's eight items to six. The second bin remains at three, while the shared construction count remains two. The three reports describe two different kinds of state; taking items is not another construction.

The counter records constructions, not how many bins remain in use. It is also state that can persist during a notebook session. Repeating an identical class definition in IJava does not reliably reset static fields. Run the ordinary examples in order; their counted classes use distinct names to keep their class-wide totals separate. Before replaying a counted example or a complete hidden answer, restart the Java kernel and run that example's full definition and caller. Retyping the class is not a reset operation.


In [ ]:
class CountedGalleryBin {
    private int stock;
    private static int created = 0;
    public CountedGalleryBin(int stock) { this.stock = stock; created = created + 1; }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) { return false; }
        stock = stock - amount;
        return true;
    }
    public static int getCreated() { return created; }
    public int getStock() { return stock; }
}
CountedGalleryBin first = new CountedGalleryBin(8);
CountedGalleryBin second = new CountedGalleryBin(3);
CountedGalleryBin alias = first;
alias.take(2);
System.out.println("First: " + first.getStock());
System.out.println("Second: " + second.getStock());
System.out.println("Created: " + CountedGalleryBin.getCreated());


Expected output:

```text
First: 6
Second: 3
Created: 2
```

Two constructor calls create the first bin at eight and the second at three and raise the shared construction count to two. The alias assignment creates no object. Its accepted take call changes only the first bin to six; the second stock and shared count remain three and two.


<details class="animation-panel" open>
<summary>Show or hide animation: instance stock and static count</summary>
<img src="media/02_encapsulation_and_shared_references/instance_stock_and_static_count.gif" alt="Track per-object fields alongside one class construction count. Print First: 6, Second: 3, Created: 2. Created counts constructor calls, not references, live objects or requests." width="960" style="max-width:100%;height:auto;">
</details>

The animation follows the state changes in the complete example above. Use **Show or hide animation** to stop viewing its motion, or [view the instance stock and static count still image](media/02_encapsulation_and_shared_references/instance_stock_and_static_count_still.png) for the final state. The reading and output explanation provide the same reasoning without animation.


## Worked Example

### Report accepted and rejected supply requests

The campus supply desk has five pens and two maps. We will represent each kind with a separate SupplyBin. The desk first asks for three pens and then four pens. Each result must report whether the request succeeded. The final report must give both remaining counts and the number of bin objects constructed.

The constructor's caller supplies valid nonnegative counts. The take operation is responsible for rejecting a request that is nonpositive or exceeds the selected bin's current stock. Our counts fit in int.


### Protect each bin and count constructions

The complete `SupplyBin` example combines the mechanisms just taught. Declare `private int stock` for each bin and `private static int binsCreated = 0` for the class. Its constructor assigns the new object's starting stock and increments the shared construction count once.

The `take` method checks `amount <= 0 || amount > stock` before subtracting. A rejected request returns `false` immediately; an accepted request subtracts and returns `true`. `getStock` returns the receiver's count, while the static `getBinsCreated` returns the class-wide total.

### Make two records and request items from one

```java
SupplyBin pens = new SupplyBin(5);
SupplyBin maps = new SupplyBin(2);
```

These two expressions create separate bins and run the constructor twice. The next two report statements request three pens and then four pens. The first request leaves two. The second exceeds that remaining stock, so it must fail without changing it. Neither call selects the maps bin.

The last three reports read each bin through its own reference, then read the shared count through `SupplyBin`. Keeping both stock reports alongside the success results lets us check that rejection preserved the pens count and that the other bin stayed separate. Run the complete class and caller below once in a fresh session for this counted class.


In [ ]:
class SupplyBin {
    private int stock;
    private static int binsCreated = 0;
    public SupplyBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
SupplyBin pens = new SupplyBin(5);
SupplyBin maps = new SupplyBin(2);
System.out.println("Take 3: " + pens.take(3));
System.out.println("Take 4: " + pens.take(4));
System.out.println("Pens: " + pens.getStock());
System.out.println("Maps: " + maps.getStock());
System.out.println("Bins: " + SupplyBin.getBinsCreated());


Expected output:

```text
Take 3: true
Take 4: false
Pens: 2
Maps: 2
Bins: 2
```

Taking three from five succeeds and leaves two pens. Taking four then fails without changing the stock. The maps bin still has two items. Both constructor calls increment the class-wide count, so two bins have been created.


## Guided Practice

Use the worked reasoning to trace, complete, modify, and repair related programs. Keep predictions before each run and record observations afterward. Empty Java cells are your programming work areas. Examples that count constructions require a fresh Java kernel before replay, because static fields may retain state in an existing session.


### Predict guarded requests and two kinds of state

Read the complete program without running it. Predict every output line. For each request, decide whether the rejection guard returns before the subtraction and what stock remains afterward. Explain what belongs to each object and what the single binsCreated field counts. Record your prediction before running the next cell or opening the answer.


In [ ]:
Your response:

Predicted complete output:

Stock before and after each request:

Accepted versus rejected request and guard reasoning:

Each object's separate state:

What binsCreated counts:


In [ ]:
class PredictionBin {
    private int stock;
    private static int binsCreated = 0;
    public PredictionBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
PredictionBin badges = new PredictionBin(6);
PredictionBin cards = new PredictionBin(3);
System.out.println("Take 2: " + badges.take(2));
System.out.println("Take 5: " + badges.take(5));
System.out.println("Badges: " + badges.getStock());
System.out.println("Cards: " + cards.getStock());
System.out.println("Bins: " + PredictionBin.getBinsCreated());


### Compare the report with a guarded-state trace

If this fixture has already run, restart the Java kernel first. Run its complete cell once and record the actual output. Identify the first matching or differing point in your prediction. Trace both requests using the response below: stock before, whether the rejection condition is true, Boolean result, and stock after. Include the separate cards stock and the shared construction count.

State the invariant and the constructor precondition. Identify the private instance field, private static field, and public operations. Explain why an outside assignment to stock is not allowed, why getStock can return a value without making the field public, and why the guard must run before subtraction. Use the Boolean and stored count together when checking rejection. The following three tasks then examine sharing through an alias, a copied parameter, and local reassignment.


In [ ]:
Your response:

Actual complete output:

First match or difference from my prediction:

Request -> badge stock before; rejection true?; returned result; badge stock after; cards stock; construction count
badges.take(2):
badges.take(5):

State invariant and constructor precondition:

Why the constructor does not promise to reject negative starting stock:

Private instance field, private static field, and public operations:

Why getStock exposes a value without exposing field assignment:

Why outside assignment is rejected:

Why rejection must precede subtraction:

Why both the Boolean and unchanged stock are evidence:

Why the other object stays separate and the counter has its value:

How my trace explains the observed report:


<details>
<summary>Show answer</summary>

The two new expressions create separate bins with stock 6 and 3. Each constructor also increments the same binsCreated field, giving a construction count of 2. Taking 2 badges is allowed, so stock falls from 6 to 4 and take returns true. Taking 5 then exceeds the remaining 4, so the early return produces false before any subtraction. The badge stock remains 4; the cards object remains at 3. The public readers expose observations without allowing callers to assign the private fields. Starting from a nonnegative count and accepting only positive amounts no larger than stock preserves the rule stock >= 0. At the first request, amount is 2, stock before the call is 6, and the rejection condition is false. At the second request, amount is 5 and stock before the call is 4, so the rejection condition is true. In both cases binsCreated remains 2 because take does not run a constructor. Each stock field belongs to one instance; binsCreated belongs to the class. Private access supports encapsulation, but the guard and the order of statements are what preserve the state rule. The constructor assumes its supplied starting count is nonnegative; it does not validate that assumption.

```java
class PredictionBin {
    private int stock;
    private static int binsCreated = 0;
    public PredictionBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
PredictionBin badges = new PredictionBin(6);
PredictionBin cards = new PredictionBin(3);
System.out.println("Take 2: " + badges.take(2));
System.out.println("Take 5: " + badges.take(5));
System.out.println("Badges: " + badges.getStock());
System.out.println("Cards: " + cards.getStock());
System.out.println("Bins: " + PredictionBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Take 5: false
Badges: 4
Cards: 3
Bins: 2
```

Common error: Subtracting an excessive request even when the method returns false. Combining the stock of separate bins. Counting take calls as constructions. Assuming private alone proves that every method preserves a valid stock count.

</details>


### Trace two references to one protected object

Predict all four output lines. Count the new expressions, identify which object original and alias can reach, and decide whether the assignment creates another bin. Trace the public take call before running. The cell includes all definitions. Restart the Java kernel before replaying this counted fixture, then run the complete cell once.


In [ ]:
Your response:

Predicted complete output:

New expressions and constructed objects:

Value copied into alias:

Object changed by alias.take(2):


In [ ]:
class AliasBin {
    private int stock;
    private static int binsCreated = 0;
    public AliasBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
AliasBin original = new AliasBin(7);
AliasBin alias = original;
System.out.println("Take 2: " + alias.take(2));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + AliasBin.getBinsCreated());


### Explain the shared result

Record the actual output and compare both reader values. Explain why a change through alias is visible through original, why the construction count differs from the number of reference variables, and why private still prevents a direct outside assignment to stock.


In [ ]:
Your response:

Actual complete output and comparison:

Why both reader values agree:

Why reference-variable count differs from construction count:

Why sharing does not remove private access control:

Kernel restart before replay, if applicable:


<details>
<summary>Show answer</summary>

There is only one new AliasBin expression. Assigning original to alias copies the reference value, so both variables reach the same object; it does not copy its fields or call a constructor. The take call through alias changes that object from 7 to 5. Both readers therefore report 5. The class counter remains 1 because creating another reference did not create another bin. The private field still changes only through the public take operation, which enforces the input rule.

```java
class AliasBin {
    private int stock;
    private static int binsCreated = 0;
    public AliasBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
AliasBin original = new AliasBin(7);
AliasBin alias = original;
System.out.println("Take 2: " + alias.take(2));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + AliasBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Original: 5
Alias: 5
Bins: 1
```

Common error: Treating alias = original as a new object construction. Copying the current stock value instead of tracing the shared object. Counting reference variables in the static construction counter.

</details>


### Trace mutation through a copied reference parameter

Predict every output line. Identify the value copied into bin at the helper call and the object reached by both bin and original. Trace the public take operation. The cell includes all definitions. Restart the Java kernel before replaying this counted fixture, then run its complete cell once.


In [ ]:
Your response:

Predicted complete output:

Value copied into the bin parameter:

Object reached by caller and parameter:

State changed inside the helper:


In [ ]:
class MutationBin {
    private int stock;
    private static int binsCreated = 0;
    public MutationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
class MutationTools {
    static void takeThroughParameter(MutationBin bin) {
        System.out.println("Take 2: " + bin.take(2));
    }
}
MutationBin original = new MutationBin(7);
MutationTools.takeThroughParameter(original);
System.out.println("Original: " + original.getStock());
System.out.println("Bins: " + MutationBin.getBinsCreated());


### Explain what survives the helper call

Record the actual report and compare it with your prediction. Explain why the caller can observe the mutation after the helper returns even though Java copied the argument value. Compare this with changing an int parameter alone in the previous lesson.


In [ ]:
Your response:

Actual complete output and comparison:

Why the caller observes the mutation:

Why the argument was still passed by value:

Difference from changing only a numeric parameter:

Kernel restart before replay, if applicable:


<details>
<summary>Show answer</summary>

When takeThroughParameter is called, Java copies the reference value held by original into the parameter bin. Both references reach the same existing MutationBin. Calling bin.take(2) changes that shared object from 7 to 5 and prints the true result. After the helper returns, original still reaches that changed object. No new expression occurs inside the helper, so binsCreated remains 1. Java passes the reference value by value; the object itself is not copied, and assigning the parameter would not assign the caller’s original variable.

```java
class MutationBin {
    private int stock;
    private static int binsCreated = 0;
    public MutationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
class MutationTools {
    static void takeThroughParameter(MutationBin bin) {
        System.out.println("Take 2: " + bin.take(2));
    }
}
MutationBin original = new MutationBin(7);
MutationTools.takeThroughParameter(original);
System.out.println("Original: " + original.getStock());
System.out.println("Bins: " + MutationBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Original: 5
Bins: 1
```

Common error: Claiming Java switches to pass by reference for objects. Assuming copying a reference duplicates the whole object. Expecting a field mutation to disappear when the parameter goes out of use.

</details>


### Trace local reassignment and a shared construction count

Predict every output line. Trace what bin reaches before and after `bin = new ReplacementBin(9)`. Decide whether original changes and whether another constructor runs. The cell includes all definitions and uses its own counted class. Restart the Java kernel before replaying it, then run the complete cell once.


In [ ]:
Your response:

Predicted complete output:

What bin reaches before reassignment:

What bin reaches afterward:

What original still reaches:

Number of constructor calls:


In [ ]:
class ReplacementBin {
    private int stock;
    private static int binsCreated = 0;
    public ReplacementBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
class ReplacementTools {
    static void replaceLocal(ReplacementBin bin) {
        bin = new ReplacementBin(9);
    }
}
ReplacementBin original = new ReplacementBin(7);
ReplacementTools.replaceLocal(original);
System.out.println("Original: " + original.getStock());
System.out.println("Bins: " + ReplacementBin.getBinsCreated());


### Separate the reference change from the construction

Record the actual output and compare it with your prediction. Explain why the original stock stays the same while the class-wide construction count changes. Compare this operation with the preceding mutation through a copied reference.


In [ ]:
Your response:

Actual complete output and comparison:

Why original still reaches unchanged stock:

Why the class counter changes:

Changing an object versus reassigning a local parameter:

Kernel restart before replay, if applicable:


<details>
<summary>Show answer</summary>

The parameter bin initially receives a copy of original’s reference. The assignment inside replaceLocal creates a second ReplacementBin with stock 9 and makes only the local parameter refer to it. It does not assign original or mutate the first object. The caller therefore still reads stock 7. The new constructor did run, so the shared construction counter becomes 2. That count records constructions, not the number of variables or objects still reachable after the helper returns. Java copies the argument value in both helper examples: mutation reaches the shared object, while reassignment replaces only the local variable’s value.

```java
class ReplacementBin {
    private int stock;
    private static int binsCreated = 0;
    public ReplacementBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
class ReplacementTools {
    static void replaceLocal(ReplacementBin bin) {
        bin = new ReplacementBin(9);
    }
}
ReplacementBin original = new ReplacementBin(7);
ReplacementTools.replaceLocal(original);
System.out.println("Original: " + original.getStock());
System.out.println("Bins: " + ReplacementBin.getBinsCreated());
```

Expected output:

```text
Original: 7
Bins: 2
```

Common error: Assuming assignment to bin also assigns the caller’s original variable. Ignoring the constructor because its reference stays local to the helper. Treating the counter as a count of variables or currently reachable objects.

</details>


### Complete a protected ticket operation

The displayed draft is incomplete and for reading only. Copy it into the empty work cell and replace all four placeholders. Choose FIELD_ACCESS from private or public so outside callers cannot assign the field. Choose REJECT_TEST from `amount <= 0 || amount > remaining` or `amount <= 0 && amount > remaining`. Choose UPDATED_REMAINING from `remaining - amount` or `remaining + amount`. Choose SUCCESS_RESULT from true or false. Keep the rest unchanged. Predict the exact request for all three tickets and the next request from the empty box. Run the completed program. Explain how the private field, public operations and guard work together, and why zero remaining is valid.

This sample is for repair:

```java
class TicketBox {
    FIELD_ACCESS int remaining;
    public TicketBox(int remaining) {
        this.remaining = remaining;
    }
    public boolean issue(int amount) {
        if (REJECT_TEST) {
            return false;
        }
        remaining = UPDATED_REMAINING;
        return SUCCESS_RESULT;
    }
    public int getRemaining() {
        return remaining;
    }
}
TicketBox tickets = new TicketBox(3);
System.out.println("Issue 3: " + tickets.issue(3));
System.out.println("Issue 1: " + tickets.issue(1));
System.out.println("Remaining: " + tickets.getRemaining());
```

Use the first response for predictions and decisions before running. Use the observation response after each run; return to the prediction response before a new version.


In [ ]:
Your response:

My four completed choices and reasons:

Predicted result and remaining tickets after each call:

Predicted complete output:


### Check the completed ticket operation

Record the actual output and compare it with your prediction. Explain why either invalid condition rejects the request, why the guard precedes the update, and why zero remaining is valid. Describe how the private field and public operations work together.


In [ ]:
Your response:

Actual complete output and comparison:

Why either invalid condition must reject:

Why the guard runs before the update:

Why zero remaining is valid:

How private state and public operations work together:


<details>
<summary>Show answer</summary>

FIELD_ACCESS is private, so outside lesson code uses the public operations instead of assigning remaining. REJECT_TEST is amount <= 0 || amount > remaining: either a nonpositive request or an excessive one must return false. UPDATED_REMAINING is remaining - amount, and SUCCESS_RESULT is true. The guard runs before subtraction. With 3 tickets, requesting 3 is valid and leaves zero. Requesting 1 then returns false without changing zero. getRemaining is a public reader; it returns the private field’s value without changing it or making the field public. The constructor receives the allowed nonnegative starting count 3.

```java
class TicketBox {
    private int remaining;
    public TicketBox(int remaining) {
        this.remaining = remaining;
    }
    public boolean issue(int amount) {
        if (amount <= 0 || amount > remaining) {
            return false;
        }
        remaining = remaining - amount;
        return true;
    }
    public int getRemaining() {
        return remaining;
    }
}
TicketBox tickets = new TicketBox(3);
System.out.println("Issue 3: " + tickets.issue(3));
System.out.println("Issue 1: " + tickets.issue(1));
System.out.println("Remaining: " + tickets.getRemaining());
```

Expected output:

```text
Issue 3: true
Issue 1: false
Remaining: 0
```

Common error: Using public for the field when direct outside assignment must be restricted. Combining rejection conditions with && and allowing one invalid condition through. Adding the issued amount instead of subtracting it. Rejecting an exact request because it leaves zero.

</details>


### Replace an alias with a separate object

This complete ModificationBin starter has the same behavior as the earlier alias check and uses a separate class name. For every run in this task, restart the Java kernel first, then run only the complete work cell once. Predict and run the starter. Change only `ModificationBin alias = original;` to `ModificationBin alias = new ModificationBin(7);`. Keep the class, other statements and variable names unchanged. Predict the result, restart the kernel and run the complete modified cell. Explain which outputs differ and why the variable name alias does not decide whether two references share an object. Then insert `System.out.println("Take 1 from original: " + original.take(1));` immediately after the Take 2 print statement. Predict the result, restart the kernel and run the whole cell again. Explain which object changes. Restore the original reference-copy assignment and remove the extra call. Restart once more before checking the restored starter.

Use the first response for predictions and decisions before running. Use the observation response after each run; return to the prediction response before a new version.


In [ ]:
Your response:

Starter predicted complete output:

Predicted complete output after the separate new expression:

Which object each variable will reach:

Predicted output after the extra original.take(1) report:

Predicted restored starter output:


In [ ]:
class ModificationBin {
    private int stock;
    private static int binsCreated = 0;
    public ModificationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
ModificationBin original = new ModificationBin(7);
ModificationBin alias = original;
System.out.println("Take 2: " + alias.take(2));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + ModificationBin.getBinsCreated());


### Explain each change in the report

After each run, record its actual complete output and compare it with that version's prediction. Explain which object each variable reaches, why the construction count changes after adding new, why the variable name alias does not establish sharing, and which object the extra original call updates. Confirm a kernel restart before each counted run and restore the original assignment and call sequence.


In [ ]:
Your response:

Starter actual output and comparison:

Separate-object actual output and comparison:

Which object each variable reaches and why:

Why the construction count changed:

Why the name alias does not determine sharing:

Extra-call actual output and changed object:

Kernel restarts performed:

Restored starter actual output:


<details>
<summary>Show answer</summary>

Replacing the reference-copy assignment with a second new ModificationBin(7) expression creates another object and invokes another constructor. The variable named alias now reaches a separate bin; its name does not force it to be an alias. Taking 2 through that reference changes only the second object to 5. original still reaches the first object with stock 7, and binsCreated is 2. The extra original.take(1) call changes only the first object to 6; the second stays at 5. Both use the same protected operations, while their instance fields stay separate. The static counter is shared and records the two constructions.

```java
class ModificationBin {
    private int stock;
    private static int binsCreated = 0;
    public ModificationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
ModificationBin original = new ModificationBin(7);
ModificationBin alias = new ModificationBin(7);
System.out.println("Take 2: " + alias.take(2));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + ModificationBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Original: 7
Alias: 5
Bins: 2
```

Common error: Assuming the name alias guarantees a shared object after its initializer changes. Creating a second object without counting its constructor. Expecting a call on one independent bin to change both stocks. Replaying the counted fixture without restarting the kernel first.

**Additional test: `Separate bins, then take 2 through alias and take 1 through original`.** The two independent stocks finish at 6 and 5; calls do not increment the shared construction count.

```java
class ModificationBin {
    private int stock;
    private static int binsCreated = 0;
    public ModificationBin(int stock) {
        this.stock = stock;
        binsCreated = binsCreated + 1;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
    public static int getBinsCreated() {
        return binsCreated;
    }
}
ModificationBin original = new ModificationBin(7);
ModificationBin alias = new ModificationBin(7);
System.out.println("Take 2: " + alias.take(2));
System.out.println("Take 1 from original: " + original.take(1));
System.out.println("Original: " + original.getStock());
System.out.println("Alias: " + alias.getStock());
System.out.println("Bins: " + ModificationBin.getBinsCreated());
```

Expected output:

```text
Take 2: true
Take 1 from original: true
Original: 6
Alias: 5
Bins: 2
```

</details>


### Reject an invalid request before changing stock

The displayed draft is intentionally incorrect. Do not run the displayed draft. It subtracts before deciding whether a request is valid. Trace both calls and predict each returned result and stock report. Identify the first statement that breaks the nonnegative-stock rule. Write the complete repaired program in the empty work cell. Check `amount <= 0 || amount > stock` before any subtraction and return false immediately for that invalid case. Only then subtract and return true. Keep the class, constructor, reader, inputs and print statements unchanged. Predict and run the repair. Then change only the two request amounts and their labels from 6 and 4 to 0 and -1; both requests must be rejected while stock stays 4. Run the complete repaired cell for that test. Explain why returning false cannot undo an earlier assignment. Restore the original requests and labels when finished.

This sample is for repair:

```java
class RequestBin {
    private int stock;
    public RequestBin(int stock) {
        this.stock = stock;
    }
    public boolean take(int amount) {
        stock = stock - amount;
        if (amount <= 0 || stock < 0) {
            return false;
        }
        return true;
    }
    public int getStock() {
        return stock;
    }
}
RequestBin bin = new RequestBin(4);
System.out.println("Take 6: " + bin.take(6));
System.out.println("Stock: " + bin.getStock());
System.out.println("Take 4: " + bin.take(4));
System.out.println("Stock: " + bin.getStock());
```

Use the first response for predictions and decisions before running. Use the observation response after each run; return to the prediction response before a new version.


In [ ]:
Your response:

Faulty call -> stock before; stock after premature subtraction; Boolean result; invariant still true?
bin.take(6):
bin.take(4):

First statement that breaks the rule:

Predicted faulty report:

Repaired guard and update order:

Predicted repaired report with requests 6 then 4:

Predicted repaired report with requests 0 then -1:


### Check rejection without a state change

Record the actual repaired report for each request pair and compare it with your prediction. Explain why returning false cannot undo an assignment, why zero and negative requests fail without changing the starting four items, and why the exact remaining request may succeed. Restore the original requests and labels and record the restored result.


In [ ]:
Your response:

Actual repaired output with requests 6 then 4:

Actual repaired output with requests 0 then -1:

Comparisons with predictions:

Why false cannot undo an earlier subtraction:

Why zero and negative requests leave stock 4:

Why the exact remaining request succeeds:

Restored original requests and actual report:


<details>
<summary>Show answer</summary>

In the faulty program, take changes stock before deciding whether to reject the request. Taking 6 from 4 stores -2 and then returns false. A later request for 4 subtracts again, stores -6 and returns false again. Returning false does not undo those assignments, and private access alone cannot prevent a badly written method from violating its own rule. The repair first checks amount <= 0 || amount > stock against the unchanged current stock. It returns false immediately for an invalid request. Only an accepted request reaches subtraction and returns true. The request for 6 now leaves stock 4; the exact request for 4 succeeds and leaves zero. Repaired zero and negative requests also return false before any subtraction, preserving stock 4.

```java
class RequestBin {
    private int stock;
    public RequestBin(int stock) {
        this.stock = stock;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
}
RequestBin bin = new RequestBin(4);
System.out.println("Take 6: " + bin.take(6));
System.out.println("Stock: " + bin.getStock());
System.out.println("Take 4: " + bin.take(4));
System.out.println("Stock: " + bin.getStock());
```

Expected output:

```text
Take 6: false
Stock: 4
Take 4: true
Stock: 0
```

Common error: Leaving subtraction before the rejection guard. Returning false after mutating the object and assuming the change is undone. Rejecting an exact request with amount >= stock. Allowing negative requests to add stock or zero requests to count as successful. Making stock public instead of fixing the operation.

**Additional test: `Zero and negative requests from an initial stock of 4`.** Both nonpositive requests return false before subtraction, so both stock reports stay at 4.

```java
class RequestBin {
    private int stock;
    public RequestBin(int stock) {
        this.stock = stock;
    }
    public boolean take(int amount) {
        if (amount <= 0 || amount > stock) {
            return false;
        }
        stock = stock - amount;
        return true;
    }
    public int getStock() {
        return stock;
    }
}
RequestBin bin = new RequestBin(4);
System.out.println("Take 0: " + bin.take(0));
System.out.println("Stock: " + bin.getStock());
System.out.println("Take -1: " + bin.take(-1));
System.out.println("Stock: " + bin.getStock());
```

Expected output:

```text
Take 0: false
Stock: 4
Take -1: false
Stock: 4
```

</details>


## Independent Practice

### Build two protected seat pools

Define SeatPool with a private int available field and a private static int poolsCreated field initialized to zero. Its public constructor accepts an initially nonnegative available count, stores it in the object and increments the class construction count. Write public boolean reserve(int seats): reject nonpositive requests or requests larger than available by returning false before changing state; otherwise subtract and return true. Add public int getAvailable() and public static int getPoolsCreated() readers. Create morning with 4 seats and evening with 2. Print the result of morning.reserve(3) labeled Reserve 3, then morning.reserve(2) labeled Reserve 2. Print the remaining Morning and Evening counts, then Pools from the static reader. The required lines are `Reserve 3: true`, `Reserve 2: false`, `Morning: 1`, `Evening: 2`, and `Pools: 2`. Plan the state rule and separate field roles, write your own complete program, predict its output, restart the Java kernel, and run the complete work cell once. Restart before each further attempt at this counted fixture. Explain what belongs to each pool and what is shared. Keep constructor input counts nonnegative; this constructor assumes that rule and does not validate negative starting counts.
Use the planning response before writing your Java program and the observation response after running.


In [ ]:
Your response:

Instance field and invariant:

Constructor precondition and initialization:

Shared static field and when it changes:

Rejection guard and update order:

Public readers and their roles:

Predicted complete output:


### Explain the two protected pools

Record the actual report and compare it with your prediction. Explain why the rejected reservation leaves Morning unchanged, why Evening stays separate, and why reservation calls do not change Pools. Identify which state belongs to each object and which state belongs to the class. Record the kernel restart used for this counted run.


In [ ]:
Your response:

Actual complete output and comparison:

Why rejection leaves Morning unchanged:

Why Evening stays separate:

Why reservations do not change Pools:

Instance state versus class state:

Kernel restart before this run:


### Test the boundary and unchanged state after rejection

Test every scenario listed below using your complete SeatPool program. Keep the class unchanged. Change only the constructor count or the two reservation amounts and their matching output labels as stated. Before every scenario and every retry, restart the Java kernel. Then run only your complete class and caller cell once. This initializes the static count as well as creating fresh morning and evening objects; repeating an identical class definition without a restart can preserve the earlier counter. Before each run, predict all five output lines; afterward record the actual results and remaining seat counts. Check the exact remaining seat, zero requests, excessive requests, negative requests and an initially empty pool. For rejected requests, verify both false and unchanged available seats. Explain why Evening and the number of constructed pools have their results in each scenario. Repair any mismatch and repeat all cases. Restore morning 4, evening 2, requests 3 then 2, and the exact original output at the end. Do not test a negative constructor count or claim that this constructor rejects one.

Use the response below to record each prediction before that run. Keep actual results in the later observation response. The required cases are listed in both responses.


In [ ]:
Your response:

Before each run, predicted five output lines:
Morning 4 / Evening 2; requests 3 then 2:

Morning 4 / Evening 2; requests 3 then 1:

Morning 4 / Evening 2; requests 0 then 5:

Morning 4 / Evening 2; requests -1 then 5:

Morning 0 / Evening 2; requests 3 then 2:


### Compare all boundary cases

Record the actual five-line report for each case and whether it matches your prediction. Explain why the exact remaining request succeeds, why zero and negative requests fail, what proves unchanged state after an excessive request, and why a pool may start empty. Account for Evening and Pools in every case. Record your restarts, corrections, and repeated cases, then restore the original inputs and report.


In [ ]:
Your response:

Actual five lines and match or repair for each case:
Morning 4 / Evening 2; requests 3 then 2:

Morning 4 / Evening 2; requests 3 then 1:

Morning 4 / Evening 2; requests 0 then 5:

Morning 4 / Evening 2; requests -1 then 5:

Morning 0 / Evening 2; requests 3 then 2:

Why exact remaining succeeds:

Why zero and negative requests fail:

Evidence of unchanged state after excess:

Why an initially empty pool is valid:

Why Evening stays separate:

Why construction count is not reservation count:

Why and when I restarted the kernel:

Corrections and repeated cases:

Actual output after restoring the original program:


<details>
<summary>Show answer</summary>

SeatPool keeps each object’s available field private and stores one shared poolsCreated field in the class. Each constructor initializes one nonnegative seat count and increments that shared count. The public reserve method rejects nonpositive or excessive requests before changing available. Morning starts at 4, accepts 3 and becomes 1; the later request for 2 is rejected without changing that 1. Evening is a separately created object and stays at 2. Two constructors ran, so the static reader reports 2 pools. Encapsulation gives callers controlled operations; the explicit guard preserves the available >= 0 state invariant. Constructor inputs are a caller precondition, not a negative-input validation implemented by this constructor. After reserving 3, an exact request for the last seat succeeds and leaves zero. A zero request and an excessive request both fail without changing the initial 4 seats. A negative request must also fail without adding seats. A pool explicitly created with zero seats is valid; every positive request from it fails and leaves zero. The separate evening pool stays at 2 in all supplied cases, and the construction count remains 2 because reservations do not construct objects. Restart the Java kernel before each case, then run the full class and caller once. Identical IJava class redefinitions can preserve static fields; replaying that definition is not a reliable counter reset.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(4);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve 3: " + morning.reserve(3));
System.out.println("Reserve 2: " + morning.reserve(2));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve 3: true
Reserve 2: false
Morning: 1
Evening: 2
Pools: 2
```

Common error: Using static for available and accidentally sharing all seat counts. Using a separate instance construction counter for each object. Changing available before returning false. Exposing a public field instead of the requested public operations. Counting reservation calls or reference variables as new pools. Claiming the constructor rejects negative values when it does not.

**Additional test: Morning 4/evening 2, reserve 3 then the exact remaining 1.** Both requests succeed; using the final seat is valid and leaves zero without changing Evening or the construction count.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(4);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve 3: " + morning.reserve(3));
System.out.println("Reserve 1: " + morning.reserve(1));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve 3: true
Reserve 1: true
Morning: 0
Evening: 2
Pools: 2
```

**Additional test: Morning 4/evening 2, reserve 0 then excessive 5.** Neither request is accepted. Both checks leave the original four morning seats unchanged.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(4);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve 0: " + morning.reserve(0));
System.out.println("Reserve 5: " + morning.reserve(5));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve 0: false
Reserve 5: false
Morning: 4
Evening: 2
Pools: 2
```

**Additional test: Morning 4/evening 2, reserve -1 then excessive 5.** A negative request is rejected without adding a seat, so the later request for 5 is still excessive and Morning stays at 4.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(4);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve -1: " + morning.reserve(-1));
System.out.println("Reserve 5: " + morning.reserve(5));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve -1: false
Reserve 5: false
Morning: 4
Evening: 2
Pools: 2
```

**Additional test: Initially empty Morning 0/evening 2, reserve 3 then 2.** Zero is an allowed initial count. Both positive requests fail; Morning stays at 0, Evening at 2, and both constructions still count.

```java
class SeatPool {
    private int available;
    private static int poolsCreated = 0;
    public SeatPool(int available) {
        this.available = available;
        poolsCreated = poolsCreated + 1;
    }
    public boolean reserve(int seats) {
        if (seats <= 0 || seats > available) {
            return false;
        }
        available = available - seats;
        return true;
    }
    public int getAvailable() {
        return available;
    }
    public static int getPoolsCreated() {
        return poolsCreated;
    }
}
SeatPool morning = new SeatPool(0);
SeatPool evening = new SeatPool(2);
System.out.println("Reserve 3: " + morning.reserve(3));
System.out.println("Reserve 2: " + morning.reserve(2));
System.out.println("Morning: " + morning.getAvailable());
System.out.println("Evening: " + evening.getAvailable());
System.out.println("Pools: " + SeatPool.getPoolsCreated());
```

Expected output:

```text
Reserve 3: false
Reserve 2: false
Morning: 0
Evening: 2
Pools: 2
```

</details>


## Summary

Encapsulation exposes intended operations while keeping state behind access controls. An invariant states the rule those operations must preserve. A copied reference can reach the same object, so mutation through an alias or parameter is visible to other references. Reassigning a local parameter does not replace the caller’s variable. Static fields belong to the class; instance fields belong separately to its objects.


### Recall the state changes without looking back

Close the answers. Explain what changes after an accepted request, what must stay unchanged after rejection, what a call through an alias can change, and what reassigning a reference parameter changes. Distinguish class state from instance state. Name one common error and a check that would expose it.


In [ ]:
Your response:

Accepted request:

Rejected request:

Call through an alias:

Reassignment of a reference parameter:

Class state versus instance state:

Common error and revealing check:


<details>
<summary>Show answer</summary>

An accepted request updates the selected object's field and reports success. A rejected request reports failure without changing that field. A call through an alias can mutate the object that other references also reach. Reassigning a reference parameter changes only that local variable; a new expression on the right can still run a constructor and affect a class counter.

An instance field belongs separately to each object. A static field belongs to the class. One common error is subtracting before validating: test an excessive request and check both false and unchanged stock. Another is counting variables as objects: identify each executed new expression and trace which object each reference reaches.

</details>


## Reflection

Describe a rule for a real object in your field, such as remaining seats or available equipment. Propose one operation and explain its result for an accepted input, a rejected input, and an exact boundary. Identify which values should belong to each object.


Write your proposed rule and the three cases in the response below.


In [ ]:
Your response:

Real object and its rule:

Proposed operation:

Accepted input and effect:

Rejected input and unchanged state:

Exact boundary and expected result:

Values belonging separately to each object:


The next lesson uses these protected operations to manage private array storage. You will keep a logical element count separate from the array capacity and grow storage while preserving the added values.


## Supplemental Reading

- [More on Java classes](https://dev.java/learn/classes-objects/more-on-classes/) covers access control and class versus instance members.
- [Calling Java methods and constructors](https://dev.java/learn/classes-objects/calling-methods-constructors/) explains why both primitive and reference argument values are copied.
